# NIHONGA — Phase 4: Insert Bridge Modules

## 1. Define and save independent residual bridge prototypes

All four measured blocks remain eligible for learned replacement. We start with the same residual bottleneck architecture for each candidate:

$$h' = h + W_{mathrm{up}},mathrm{GELU}(W_{mathrm{down}}h).$$

The model's saved configuration gives a hidden width of **4096**. We use a bottleneck of **256**, with bias-free projections and a separate set of weights for each block. The bridge predicts a residual update for each token. It is a learned approximation, not a weighted average of adjacent blocks. The unconditioned baseline may not reproduce attention or timestep-dependent behavior sufficiently; evaluation determines whether it needs more capacity or conditioning.

The up projection starts at zero, so the initial output equals the input. Training will change those weights to approximate the teacher block's input-to-output transformation. Identity initialization does not mean we intend to keep identity as the final replacement. Only initialized weights are saved in this step; a disposable synthetic gradient check verifies that learning can move the output away from identity, without being mistaken for teacher training.

We save the implementation, experiment manifest, four initial checkpoints and check results locally. Completed reruns validate their hashes and read cached results without importing Torch or repeating computations. No teacher is loaded. Training targets must come from Phase 1 TRAIN; validation stays separate, and internal test, Phase 3 probes and Qwen-Image-Bench remain evaluation-only. This core module needs a model-specific wrapper before pipeline insertion because the transformer inspects each block's attention processor and uses prefix caches. We retain the frozen 32-block teacher unchanged here.


## Before and after: what the learned bridge replaces

Block 5 is shown as an example. Blocks **2, 3, 4 and 5** each have an independent bridge candidate, tested separately. The intended replacement contains **31 original transformer blocks plus one learned bridge**. Currently only initial bridge weights are saved; no bridge has been trained on teacher targets or installed in the pipeline.

```text
BEFORE: intact teacher             AFTER: intended single-block replacement

Noisy latent + prompt + timestep   Noisy latent + prompt + timestep
              |                                 |
              v                                 v
        [ Block 0 ]                       [ Block 0 ]
              |                                 |
             ...                               ...
              |                                 |
        [ Block 4 ]                       [ Block 4 ]
              |                                 |
        [ Block 5 ]                   [ Learned bridge ]
              |                                 |
        [ Block 6 ]                       [ Block 6 ]
              |                                 |
             ...                               ...
              |                                 |
        [ Block 31 ]                      [ Block 31 ]
              |                                 |
        [ Output head ]                   [ Output head ]
              |                                 |
              v                                 v
      Predicted velocity                Predicted velocity
```

The bridge approximates the missing block's hidden-state update. The output head still predicts velocity, which the scheduler uses to update the noisy latent at each denoising step.

### Inside the residual bottleneck bridge

```text
                     h: 4096 components
                             |
                +------------+-------------+
                |                          |
                |                   [ Down projection ]
                |                      4096 -> 256
                |                          |
                |                        [ GELU ]
                |                          |
                |                    [ Up projection ]
                |                      256 -> 4096
                |                          |
                |                   Predicted update
                |                          |
                +------------( + )---------+
                              |
                              v
                     h' = h + update
```

The direct branch preserves the input. The learned branch predicts the residual to add, independently for each token. At initialization its up projection is zero, making the update zero; teacher-supervised training must learn a useful nonzero update. Each bridge has 2,097,152 parameters. Whether it can reproduce the original block sufficiently well remains an experimental question.

### How teacher supervision trains the bridge

```text
                        Same block input h
                                |
                  +-------------+-------------+
                  |                           |
          [ Teacher block 5 ]         [ Trainable bridge ]
                frozen                        |
                  |                           |
                  v                           v
            Target output               Bridge output
                  |                           |
                  +----- compare outputs -----+
                                |
                         Hidden-state loss
                                |
                     Update bridge weights only
```

Teacher block inputs and outputs are captured from the training split and saved. The teacher remains frozen; only the bridge learns. Validation examples stay separate. Matching a block's hidden output is the fitting objective, while held-out velocity predictions and full generated images determine whether the replacement preserves the model's behavior.


In [1]:
# Define and serialize independent, trainable bridge prototypes without loading the teacher.
import hashlib
import json
from pathlib import Path

root = Path('data/phase4')
root.mkdir(parents=True, exist_ok=True)
architecture_path = Path('data/phase3/architecture.json')
architecture = json.loads(architecture_path.read_text(encoding='utf-8'))
config = json.loads(Path('data/phase3/model_config.json').read_text(encoding='utf-8'))
if architecture['config'] != config:
    raise RuntimeError('Saved architecture metadata disagrees with model configuration.')
width = config['num_attention_heads'] * config['attention_head_dim']
shortlist_path = Path('data/phase3/bypass_shortlist.json')
shortlist = json.loads(shortlist_path.read_text(encoding='utf-8'))
candidates = shortlist['candidate_layers']
if set(candidates) != {2, 3, 4, 5} or config['num_layers'] != 32 or width != 4096:
    raise RuntimeError('Architecture or measured candidate batch changed; preserve existing artifacts.')

definition = '''import torch
from torch import nn

class ResidualBottleneckBridge(nn.Module):
    """Trainable tokenwise residual approximation; pipeline integration is a separate wrapper."""
    def __init__(self, hidden_dim=4096, bottleneck_dim=256):
        super().__init__()
        self.down = nn.Linear(hidden_dim, bottleneck_dim, bias=False)
        self.activation = nn.GELU()
        self.up = nn.Linear(bottleneck_dim, hidden_dim, bias=False)
        # Identity initialization is a starting point for learning, not the final replacement.
        nn.init.zeros_(self.up.weight)

    def forward(self, hidden_states):
        return hidden_states + self.up(self.activation(self.down(hidden_states)))
'''
definition_hash = hashlib.sha256(definition.encode()).hexdigest()
sources = {'architecture.json': hashlib.sha256(architecture_path.read_bytes()).hexdigest(),
           'bypass_shortlist.json': hashlib.sha256(shortlist_path.read_bytes()).hexdigest()}
for layer in candidates:
    path = Path(f'data/phase3/bypass/block_{layer:02d}_pilot/summary.json')
    raw = path.read_bytes()
    source = json.loads(raw)
    if source['bypassed_layer'] != layer or source['n_comparisons'] != 15:
        raise RuntimeError('Candidate bypass evidence is incomplete.')
    sources[f'block_{layer:02d}_pilot/summary.json'] = hashlib.sha256(raw).hexdigest()
image_review_path = Path('data/phase3/bypass/full_images_pilot/visual_review.json')
sources['full_images_pilot/visual_review.json'] = hashlib.sha256(image_review_path.read_bytes()).hexdigest()
split_paths = {name: Path(f'data/phase1/pilot_20k/{name}.jsonl') for name in ('train','validation','internal_test')}
split_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name,path in split_paths.items()}
request = {
    'version': 'phase4-bridge-prototypes-v1',
    'model_repo': architecture['model_repo'], 'model_revision': architecture['model_revision'],
    'config_sha256': architecture['config_sha256'], 'num_teacher_blocks': config['num_layers'],
    'hidden_dim': width, 'bottleneck_dim': 256, 'bias': False,
    'formula': 'h + W_up(GELU(W_down(h)))', 'definition_sha256': definition_hash,
    'initialization': 'random down projection, zero up projection; initially identity, trained later',
    'initialization_seed_by_layer': {str(layer): 20260928 + layer for layer in candidates},
    'prototype_dtype': 'float32', 'candidate_layers': candidates,
    'candidate_policy': 'all four measured candidates remain eligible for learned bridges; no layer permanently excluded by identity tests',
    'replacement_policy': 'independent single-block experiments before joint replacement; same rank and training budget per candidate',
    'phase3_source_sha256': sources, 'split_sha256': split_hashes,
    'training_source': str(split_paths['train']), 'selection_source': str(split_paths['validation']),
    'evaluation_only': ['internal_test', 'qwen_image_bench', 'phase3 pilot and full probe sets'],
    'target_capture_policy': 'capture paired block input and output from TRAIN; validation separately; save token roles, masks and stage metadata',
    'integration_policy': 'core module is not a complete drop-in block; preserve parent processor inspection and rebuild downstream prefix caches with the bridge',
    'conditioning_policy': 'unconditioned tokenwise baseline; evaluate prefix and image tokens and each stage separately; timestep conditioning only if needed',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest_path = root / 'bridge_prototypes_request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Saved bridge experiment differs; preserve checkpoints and inspect.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
module_path = root / 'bridge_definition.py'
if module_path.exists():
    if module_path.read_bytes() != definition.encode():
        raise RuntimeError('Saved bridge implementation changed.')
else:
    with module_path.open('xb') as handle:
        handle.write(definition.encode())

summary_path = root / 'bridge_prototypes_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if summary['request_key'] != key or summary['definition_sha256'] != definition_hash:
        raise RuntimeError('Prototype summary belongs to different inputs.')
    for layer, digest in summary['checkpoint_sha256'].items():
        path = root / f'bridge_block_{int(layer):02d}_initial.pt'
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError('Saved initial bridge checkpoint changed or is missing.')
    print('Reusing four verified bridge prototypes and saved checks; no model imports or computations.')
else:
    import math
    import torch
    namespace = {}
    exec(compile(definition, str(module_path), 'exec'), namespace)
    Bridge = namespace['ResidualBottleneckBridge']
    records, hashes = [], {}
    for layer in candidates:
        path = root / f'bridge_block_{layer:02d}_initial.pt'
        seed = request['initialization_seed_by_layer'][str(layer)]
        if path.exists():
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['request_key'] != key or checkpoint['layer'] != layer:
                raise RuntimeError('Existing prototype checkpoint has different provenance.')
        else:
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(seed)
                bridge = Bridge(width, request['bottleneck_dim'])
            checkpoint = {'request_key': key, 'layer': layer, 'initialization_seed': seed,
                          'trained': False, 'state_dict': bridge.state_dict()}
            temporary = path.with_suffix('.tmp')
            torch.save(checkpoint, temporary)
            temporary.replace(path)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed + 1000)
            bridge = Bridge(width, request['bottleneck_dim'])
            bridge.load_state_dict(checkpoint['state_dict'], strict=True)
            h = torch.randn(1, 6, width)
            # A nonzero synthetic target checks trainability; it is not teacher supervision.
            target = h + 0.1 * torch.randn_like(h)
        prediction = bridge(h)
        if not torch.equal(prediction, h):
            raise RuntimeError('Zero-up initialization is not exactly identity.')
        loss = (prediction - target).square().mean()
        loss.backward()
        grad_norm = float(torch.linalg.vector_norm(bridge.up.weight.grad))
        if not math.isfinite(grad_norm) or grad_norm <= 0:
            raise RuntimeError('The up projection receives no usable learning signal.')
        with torch.no_grad():
            bridge.up.weight.add_(bridge.up.weight.grad, alpha=-0.1)
            changed = bridge(h)
            loss_after = float((changed - target).square().mean())
        if not torch.isfinite(changed).all() or torch.equal(changed, h) or loss_after >= float(loss.detach()):
            raise RuntimeError('Synthetic learning check failed.')
        parameter_count = sum(p.numel() for p in bridge.parameters())
        if parameter_count != 2 * width * request['bottleneck_dim']:
            raise RuntimeError('Bridge parameter budget differs from definition.')
        hashes[str(layer)] = hashlib.sha256(path.read_bytes()).hexdigest()
        records.append({'layer': layer, 'initialization_seed': seed, 'parameters': parameter_count,
                        'identity_at_initialization': True, 'synthetic_up_gradient_norm': grad_norm,
                        'synthetic_loss_before': float(loss.detach()), 'synthetic_loss_after_one_update': loss_after,
                        'synthetic_nonidentity_after_update': True, 'checkpoint_status': 'initialization only; synthetic update not saved'})
    summary = {'request_key': key, 'definition_sha256': definition_hash, 'checkpoint_sha256': hashes,
               'checks': records, 'parameters_per_bridge': 2*width*request['bottleneck_dim'],
               'full_linear_parameters': width*width,
               'status': 'prototypes initialized; no teacher targets collected, no bridge trained or installed'}
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Measured bridge candidates:', candidates)
print(f"Actual hidden width: {width}; bottleneck: {request['bottleneck_dim']}.")
print(f"Parameters per bridge: {summary['parameters_per_bridge']:,}; full residual linear map: {summary['full_linear_parameters']:,}.")
print('Four independent initial checkpoints saved. Synthetic checks show learning can move each bridge away from identity.')
print('Training: Phase 1 TRAIN; selection: VALIDATION. Internal test, Phase 3 evaluation prompts and benchmark remain evaluation-only.')
print(summary['status'])
print('Artifacts:', root)


Measured bridge candidates: [2, 4, 5, 3]
Actual hidden width: 4096; bottleneck: 256.
Parameters per bridge: 2,097,152; full residual linear map: 16,777,216.
Four independent initial checkpoints saved. Synthetic checks show learning can move each bridge away from identity.
Training: Phase 1 TRAIN; selection: VALIDATION. Internal test, Phase 3 evaluation prompts and benchmark remain evaluation-only.
prototypes initialized; no teacher targets collected, no bridge trained or installed
Artifacts: data\phase4


## What the bridge prototype results mean

**Blocks 2, 3, 4 and 5 all remain candidates.** We have defined the same trainable replacement architecture for each and saved four separate initial checkpoints. No candidate is excluded because identity harmed an image. Other blocks can also be investigated; these four are the initial measured batch. Being a candidate does not guarantee that a sufficiently cheap bridge will recover its behavior.

The teacher's hidden vector has **4096 components**. Each bridge compresses that vector to **256 components**, applies GELU, maps back to 4096 components, and adds the predicted update to the input. It operates independently on each token. The bottleneck is a capacity choice to test, not an established optimum.

| Quantity | Value | Meaning |
|---|---:|---|
| Candidate bridges | 4 | Independent experiments, not four simultaneous replacements |
| Hidden width | 4096 | Actual model width derived from the frozen configuration |
| Bottleneck width | 256 | Size of the bridge's intermediate representation |
| Parameters per bottleneck bridge | 2,097,152 | Two bias-free projection matrices |
| Parameters in a full residual linear map | 16,777,216 | A single 4096 × 4096 matrix for comparison |
| Bottleneck / full linear parameter ratio | 12.5% | One eighth as many parameters as the full linear alternative |
| Initial checkpoint tensor payload | 8 MiB per bridge | FP32 weights; excludes serialization overhead and training memory |

The parameter ratio compares two bridge architectures, not a bridge with the entire transformer block. This step measures no model speedup or quality improvement. Fewer parameters do not by themselves prove faster end-to-end generation.

**Why initialization still behaves like identity:** the down projection is random, while the up projection starts at zero. Initially the bridge predicts a zero residual, so its output is exactly the input. The up projection receives a nonzero gradient under the synthetic check; after one disposable update, the output changes and the synthetic loss decreases. Training on teacher targets must learn a useful residual. Identity is only the initialization.

The synthetic checks below test the implementation's ability to learn, not its ability to replace a teacher block. They use random tensors without a teacher or prompts. Those disposable updates were not saved: all four persisted checkpoints still have zero up projections and are explicitly marked untrained.

| Block | Exactly identity at initialization | Nonidentity after synthetic update | Synthetic loss before | Synthetic loss after |
|---|---|---|---:|---:|
| 2 | Yes | Yes | 0.01012461 | 0.01011989 |
| 4 | Yes | Yes | 0.00988936 | 0.00988416 |
| 5 | Yes | Yes | 0.00992399 | 0.00991924 |
| 3 | Yes | Yes | 0.01004768 | 0.01004226 |

**Data separation:** paired teacher block inputs and outputs must be collected from Phase 1 TRAIN. Validation is separate and used for selection. The existing Phase 3 probe prompts, including the astronaut image that informed our discussion, remain evaluation examples; their saved tensors are not bridge-training data. Qwen-Image-Bench is evaluation-only. Token roles, masks and stage metadata must be retained when targets are captured, because the model processes prefix tokens at the first step and image tokens at later cached steps.

**What is ready:** a serialized implementation, a reproducible experiment manifest, four independent initial checkpoints and saved local check results. Checkpoint inspection confirms correct tensor shapes, finite and distinct down projections, and zero up projections. A completed rerun imported no Torch, repeated no checks or calculations, called no API and changed no artifact.

**What is not established yet:** no teacher training targets have been collected for these bridges, no teacher-supervised training has run, and no bridge has been inserted into the full model. The core module is not yet a complete drop-in transformer block: pipeline insertion needs to preserve processor inspection and ensure downstream prefix caches are generated with the bridge. The baseline bridge has no attention or explicit timestep conditioning; evaluation must determine whether that limitation prevents sufficient recovery. Hidden-state fitting, held-out velocity checks, full-image prompt adherence and matched runtime measurements are all required before accepting a replacement.

Artifacts are saved in `data/phase4/`: `bridge_definition.py`, `bridge_prototypes_request.json`, `bridge_prototypes_summary.json`, and one `bridge_block_XX_initial.pt` checkpoint per candidate. The frozen teacher and Phase 3 diagnostic artifacts are preserved.


## 2. Implement and verify the transformer-block wrapper

The core bridge needs the same forward arguments as the teacher block. We wrap it with that interface and a parameter-free processor reference for the parent model's mask inspection. The replacement stays at the original block index, preserving modulation and cache indices. Its forward uses only the tokenwise bridge; it has no attention projections or K/V cache.

We verify all four saved 4096-wide prototypes, then insert a nonzero synthetic bridge into a tiny three-block model using the exact [pinned upstream transformer source](https://raw.githubusercontent.com/huggingface/diffusers/bdc2bea37a36038c44452811610489ea30ede229/src/diffusers/models/transformers/transformer_qwenimage21.py), whose checksum must match Phase 3. Local supporting package versions are recorded. For both text-only and condition-image inputs with padded text, we compare cached target outputs with a fresh full forward at a changed timestep and changed target latent. We also check that downstream prefix keys change when the bridge changes, and that positional calls support gradients.

Insertion returns a fresh empty cache. Callers must discard earlier caches and extract the student's prefix after insertion and after weight changes. The helper does not make teacher caches safe to reuse. The tiny model checks software compatibility, not learned behavior, full-pipeline quality, BF16 GPU execution, or speed. Completed reruns read saved reports and hashes without repeating forwards or downloads. Initial bridge checkpoints are preserved.


In [2]:
# Persist and exercise the block wrapper once; completed reruns only read saved artifacts.
import hashlib
import json
from pathlib import Path

root = Path('data/phase4')
wrapper_source = '"""Qwen-Image 2.1 block contract for an unconditioned tokenwise bridge."""\nfrom types import SimpleNamespace\nfrom torch import nn\n\n\nclass QwenImage21BridgeBlock(nn.Module):\n    def __init__(self, bridge, processor):\n        super().__init__()\n        self.bridge = bridge\n        # Parent forward inspects this processor to choose mask construction.\n        # This contains no attention projections and performs no attention.\n        self.attn = SimpleNamespace(processor=processor)\n\n    def forward(self, hidden_states, modulation, rotary_emb=None,\n                attention_mask=None, target_token_mask=None, layer_cache=None,\n                kv_cache_mode=None, cache_write_slice=None, segments=None,\n                key_valid=None):\n        if kv_cache_mode not in (None, \'extract\', \'cached\'):\n            raise ValueError(\'Unknown KV cache mode.\')\n        # No K/V is required in this slot. Subsequent surviving blocks build\n        # their own prefix K/V from the transformed prefix during extraction.\n        return self.bridge(hidden_states)\n\n\ndef insert_bridge(transformer, layer, bridge, cache_class):\n    """Replace one slot; caller must discard all prior caches and use returned cache."""\n    if not 0 <= layer < len(transformer.transformer_blocks):\n        raise IndexError(\'Bridge slot is outside transformer depth.\')\n    if bridge.down.in_features != transformer.config.num_attention_heads * transformer.config.attention_head_dim:\n        raise ValueError(\'Bridge width differs from transformer width.\')\n    old = transformer.transformer_blocks[layer]\n    reference = next(transformer.parameters())\n    bridge = bridge.to(device=reference.device, dtype=reference.dtype)\n    wrapper = QwenImage21BridgeBlock(bridge, old.attn.processor)\n    wrapper.train(transformer.training)\n    transformer.transformer_blocks[layer] = wrapper\n    return old, cache_class(len(transformer.transformer_blocks))\n'
wrapper_path = root / 'bridge_block_definition.py'
source_path = root / 'transformer_qwenimage21_pinned.py'
source_url = 'https://raw.githubusercontent.com/huggingface/diffusers/bdc2bea37a36038c44452811610489ea30ede229/src/diffusers/models/transformers/transformer_qwenimage21.py'
expected_source_hash = '0eb0555e21ca93195e1fe9389113cafbf0e8822f6454c3001cebb3d21521ebd7'
if wrapper_path.exists():
    if wrapper_path.read_bytes() != wrapper_source.encode():
        raise RuntimeError('Saved wrapper differs; preserve and inspect.')
else:
    with wrapper_path.open('xb') as handle:
        handle.write(wrapper_source.encode())
if not source_path.exists():
    import urllib.request
    raw = urllib.request.urlopen(source_url, timeout=60).read()
    if hashlib.sha256(raw).hexdigest() != expected_source_hash:
        raise RuntimeError('Pinned upstream source does not match Phase 3.')
    with source_path.open('xb') as handle:
        handle.write(raw)
if hashlib.sha256(source_path.read_bytes()).hexdigest() != expected_source_hash:
    raise RuntimeError('Saved upstream transformer source changed.')
prototype_manifest = json.loads((root / 'bridge_prototypes_request.json').read_text())
prototype_summary = json.loads((root / 'bridge_prototypes_summary.json').read_text())
if prototype_summary['request_key'] != prototype_manifest['request_key']:
    raise RuntimeError('Prototype provenance differs.')
if hashlib.sha256((root / 'bridge_definition.py').read_bytes()).hexdigest() != prototype_summary['definition_sha256']:
    raise RuntimeError('Core bridge implementation changed.')
for layer, digest in prototype_summary['checkpoint_sha256'].items():
    if hashlib.sha256((root / f'bridge_block_{int(layer):02d}_initial.pt').read_bytes()).hexdigest() != digest:
        raise RuntimeError('Initial checkpoint changed.')
request = {
    'version': 'phase4-block-contract-v1',
    'prototype_request_key': prototype_manifest['request_key'],
    'checkpoint_sha256': prototype_summary['checkpoint_sha256'],
    'wrapper_sha256': hashlib.sha256(wrapper_source.encode()).hexdigest(),
    'transformer_source_sha256': expected_source_hash, 'source_url': source_url,
    'seed': 20260928, 'test_dtype': 'float32',
    'tiny_architecture': {'num_layers': 3, 'num_attention_heads': 2, 'attention_head_dim': 16,
                          'context_in_dim': 32, 'in_channels': 4, 'out_channels': 4,
                          'axes_dims_rope': [4, 6, 6], 'mlp_ratio': 3, 'causal_condition': True},
    'tiny_replaced_slot': 1, 'tiny_bottleneck': 8,
    'scope': 'CPU synthetic block contract; pinned transformer source with local supporting dependencies; no teacher weights or quality evaluation',
    'cache_policy': 'preserve original slot indices; discard old cache and extract new student prefix; bridge slot has no K/V',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest_path = root / 'bridge_block_contract_request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != manifest:
        raise RuntimeError('Contract request changed.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
summary_path = root / 'bridge_block_contract_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    if summary['request_key'] != key or summary['status'] != 'passed':
        raise RuntimeError('Saved contract checks failed or differ; do not repeat automatically.')
    print('Reusing saved contract checks; no Torch import, downloads, model forwards or API calls.')
else:
    import importlib.util
    import importlib.metadata
    import inspect
    import sys
    import torch
    def load_module(name, path):
        spec = importlib.util.spec_from_file_location(name, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
        return module
    core = load_module('nihonga_bridge_core', root / 'bridge_definition.py')
    adapter = load_module('nihonga_bridge_block', wrapper_path)
    # Package name preserves the upstream source's relative imports.
    pinned = load_module('diffusers.models.transformers.nihonga_qwenimage21_pinned', source_path)
    summary = {'request_key': key, 'status': 'failed', 'packages': {
        name: importlib.metadata.version(name) for name in ('torch', 'diffusers', 'transformers')}}
    try:
        expected_signature = inspect.signature(pinned.QwenImage21TransformerBlock.forward)
        actual_signature = inspect.signature(adapter.QwenImage21BridgeBlock.forward)
        if list(expected_signature.parameters) != list(actual_signature.parameters):
            raise RuntimeError('Wrapper argument order differs from pinned block.')
        full_width_checks = []
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(request['seed'])
            for layer in prototype_manifest['request']['candidate_layers']:
                checkpoint = torch.load(root / f'bridge_block_{layer:02d}_initial.pt', weights_only=True, map_location='cpu')
                bridge = core.ResidualBottleneckBridge(4096, 256)
                bridge.load_state_dict(checkpoint['state_dict'], strict=True)
                block = adapter.QwenImage21BridgeBlock(bridge, pinned.QwenImage21AttnProcessor())
                h = torch.randn(1, 5, 4096)
                with torch.no_grad():
                    output = block(h, torch.empty(0))
                assert torch.equal(h, output)
                assert sum(p.numel() for p in block.parameters()) == 2097152
                full_width_checks.append({'layer': layer, 'parameters': 2097152, 'exact_identity': True})
            model = pinned.QwenImage21Transformer2DModel(**request['tiny_architecture']).eval()
            bridge = core.ResidualBottleneckBridge(32, 8)
            # Nonzero synthetic weights expose stale-cache mistakes that identity could hide.
            torch.nn.init.normal_(bridge.up.weight, std=0.1)
            old, cache = adapter.insert_bridge(model, 1, bridge, pinned.QwenImage21KVCache)
            assert len(model.transformer_blocks) == 3
            assert model.transformer_blocks[1].attn.processor is old.attn.processor
            assert sum(p.numel() for p in model.transformer_blocks[1].parameters()) == 512
            records = []
            with torch.no_grad():
                for condition_image in (False, True):
                    text_len = 3 if not condition_image else 4
                    image_tokens = 4 if not condition_image else 8
                    img_mask = torch.tensor([[False, False, False, True]] if not condition_image
                                            else [[False, True, False, False, True]])
                    text = torch.randn(1, text_len, 32)
                    text_mask = torch.tensor([[True, True, False]] if not condition_image
                                             else [[True, True, True, False]])
                    shapes = [[(1, 2, 2)]] if not condition_image else [[(1, 2, 2), (1, 2, 2)]]
                    latents = torch.randn(1, image_tokens, 4)
                    common = dict(encoder_hidden_states=text, encoder_hidden_states_mask=text_mask,
                                  img_mask=img_mask, img_shapes=shapes)
                    cache = pinned.QwenImage21KVCache(3)
                    model(hidden_states=latents, timestep=torch.tensor([0.9]),
                          kv_cache=cache, kv_cache_mode='extract', **common)
                    assert cache.get_layer(1).k is None and cache.get_layer(1).v is None
                    assert all(cache.get_layer(i).k is not None for i in (0, 2))
                    changed = latents.clone()
                    changed[:, -4:] = torch.randn_like(changed[:, -4:])
                    fresh = model(hidden_states=changed, timestep=torch.tensor([0.4]), **common).sample[:, -4:]
                    cached = model(hidden_states=changed, timestep=torch.tensor([0.4]),
                                   kv_cache=cache, kv_cache_mode='cached', **common).sample
                    error = float((fresh - cached).abs().max())
                    assert torch.allclose(fresh, cached, atol=2e-6, rtol=2e-5)
                    # Ensure the downstream prefix actually depends on the nonzero bridge.
                    saved_up = bridge.up.weight.clone()
                    bridge.up.weight.zero_()
                    identity_cache = pinned.QwenImage21KVCache(3)
                    model(hidden_states=latents, timestep=torch.tensor([0.9]),
                          kv_cache=identity_cache, kv_cache_mode='extract', **common)
                    delta = float((cache.get_layer(2).k - identity_cache.get_layer(2).k).abs().max())
                    bridge.up.weight.copy_(saved_up)
                    assert delta > 0
                    records.append({'condition_image': condition_image, 'padded_text': True,
                                    'cached_vs_fresh_max_abs_error': error,
                                    'downstream_prefix_key_max_change': delta,
                                    'bridge_slot_cache_empty': True})
            # Check the positional call used by gradient checkpointing and trainable gradient flow.
            h = torch.randn(1, 5, 32, requires_grad=True)
            out = model.transformer_blocks[1](h, torch.empty(0), None, None, None, None, None, None, None, None)
            out.square().mean().backward()
            assert bridge.up.weight.grad is not None and torch.isfinite(bridge.up.weight.grad).all()
            assert float(bridge.up.weight.grad.norm()) > 0
            summary.update(status='passed', full_width_checks=full_width_checks, cache_checks=records,
                           positional_forward_and_gradient_passed=True,
                           full_model_installed=False, teacher_trained=False)
    except Exception as error:
        summary['error'] = f'{type(error).__name__}: {error}'
        with summary_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(summary, indent=2) + '\n')
        raise
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
print('Block contract:', summary['status'])
print('Verified full-width initial wrappers:', len(summary['full_width_checks']))
for row in summary['cache_checks']:
    print('Condition image:', row['condition_image'],
          '| cached/fresh max error:', row['cached_vs_fresh_max_abs_error'],
          '| downstream prefix key change:', row['downstream_prefix_key_max_change'])
print('Only bridge projection parameters remain in a replacement slot. No teacher weights loaded.')


Reusing saved contract checks; no Torch import, downloads, model forwards or API calls.
Block contract: passed
Verified full-width initial wrappers: 4
Condition image: False | cached/fresh max error: 5.960464477539063e-08 | downstream prefix key change: 0.4651245176792145
Condition image: True | cached/fresh max error: 0.0 | downstream prefix key change: 0.4150390625
Only bridge projection parameters remain in a replacement slot. No teacher weights loaded.


## What the block-wrapper results mean

**The wrapper passes the local software contract checks.** All four saved initial bridges load into replacement slots with **2,097,152 parameters each**, exactly the two projection matrices. Their outputs are exactly identity because their saved up projections are still zero. The wrapper retains no teacher attention or feed-forward parameters.

The cache experiment uses the checksum-verified Phase 3 transformer implementation with **three random blocks, hidden width 32 and bottleneck width 8**. A nonzero synthetic bridge replaces slot 1. The original slot count and modulation indexing remain unchanged. This small model makes the cache checks affordable; it is not the pretrained teacher or a compressed production student.

| Synthetic input | Maximum absolute cached/fresh target-output difference | Maximum downstream prefix-key change when bridge is made identity |
|---|---:|---:|
| Text only, including padded text | 0.0000000596 | 0.4651 |
| Text plus condition image, including padded text | 0.0 | 0.4150 |

The first metric compares two ways of running the same tiny student: reuse its freshly extracted prefix versus recompute the full sequence, after changing the target latent and timestep. The text-only difference is about six hundred-millionths; the condition-image comparison is exactly equal in this run. Both pass the saved FP32 tolerance of absolute 0.000002 and relative 0.00002. This supports using the wrapper with freshly constructed student caches in the tested layouts. It is not a teacher/student error or an image-quality score.

The second metric measures how much a surviving downstream block's stored keys change when the bridge's synthetic nonzero update is removed. Values **0.4651 and 0.4150** show that the downstream prefix depends on the bridge. Their magnitudes have no quality interpretation. They support the requirement to discard old teacher or student caches when inserting a bridge or changing its weights, then extract new prefix keys from the student. The bridge's own cache slot stays empty because it performs no attention; surviving blocks on both sides populate their original slots.

The wrapper's argument names and order match the pinned block, including positional arguments used by gradient checkpointing. A direct positional forward/backward gives a finite nonzero bridge gradient. This establishes that the wrapper remains trainable; a full training run with checkpointing enabled has not been tested.

**Decision supported:** the wrapper is ready for a full-model insertion experiment. **Still uncertain:** BF16 GPU behavior, pipeline-level execution, mixed/flex attention configuration, learned approximation quality, and measured latency or memory savings. The parameter-free processor reference supports the parent's current forward inspection; generic attention-processor replacement utilities, compilation and distributed training are not certified by these checks. This step neither trains bridges nor modifies the frozen teacher or any saved initial checkpoint.

Artifacts: `data/phase4/bridge_block_definition.py`, `bridge_block_contract_request.json`, `bridge_block_contract_summary.json`, and `transformer_qwenimage21_pinned.py` (upstream license header preserved). The report records local supporting dependency versions. A completed rerun of both phase 4 code cells imported no Torch, Diffusers, Modal or network client and changed no saved artifact. Checks and results are reused locally.


## 3. Insert block 5 into the full pinned pipeline

We assemble one independent student on the existing Modal H100 runtime by loading the pinned teacher, replacing original slot 5 with its saved BF16 bridge wrapper, and preserving the other 31 transformer blocks and all original slot indices. Inherited weights stay frozen and only bridge parameters are marked trainable. No training occurs. Block 5 is an integration example; all four candidates remain eligible.

We run the saved meadow evaluation prompt with its original seed, resolution, scheduler and 40 steps. The pipeline constructs fresh student prefix caches. A hook verifies an exactly identity bridge at every BF16 call, including one prefix extraction and subsequent cached steps. We compare pixels with the existing block-5 identity-bypass image, without rerunning the teacher. This is an initialization consistency check, not a quality score.

The image, parameter counts, call metadata, diagnostic timing and memory, and a reconstruction recipe are saved locally and backed up on the results volume. The recipe combines the pinned teacher, unchanged initial checkpoint and checksummed wrapper without copying teacher weights. Completed local reruns make no Modal calls. Remote completed reports are recovered before scheduling a GPU; a started attempt without a report stops rather than repeating work.


In [3]:
# Insert one saved bridge into the pinned BF16 pipeline and checkpoint a single smoke result.
import base64
import hashlib
import json
from pathlib import Path

root = Path('data/phase4')
run_dir = root / 'insertion_block_05_smoke'
run_dir.mkdir(exist_ok=True)
contract_raw = (root / 'bridge_block_contract_summary.json').read_bytes()
contract = json.loads(contract_raw)
if contract['status'] != 'passed':
    raise RuntimeError('Local block contract has not passed.')
prototypes = json.loads((root / 'bridge_prototypes_summary.json').read_text())
prototype_request = json.loads((root / 'bridge_prototypes_request.json').read_text())
prior = json.loads(Path('data/phase3/bypass/full_images_pilot/request.json').read_text())
target = prior['request']['target_request']
row = target['rows'][0]
reference_path = Path('data/phase3/bypass/full_images_pilot') / f"{row['eval_id']}-block-05.png"
reference_report = json.loads(reference_path.with_suffix('.json').read_text())
reference_hash = hashlib.sha256(reference_path.read_bytes()).hexdigest()
if reference_hash != reference_report['image_sha256'] or reference_report['job'] != {'row': row, 'layer': 5}:
    raise RuntimeError('Saved identity-bypass reference differs.')
checkpoint_path = root / 'bridge_block_05_initial.pt'
checkpoint_bytes = checkpoint_path.read_bytes()
checkpoint_hash = hashlib.sha256(checkpoint_bytes).hexdigest()
if checkpoint_hash != prototypes['checkpoint_sha256']['5']:
    raise RuntimeError('Initial checkpoint changed.')
core_source = (root / 'bridge_definition.py').read_text(encoding='utf-8')
wrapper_source = (root / 'bridge_block_definition.py').read_text(encoding='utf-8')
contract_request = json.loads((root / 'bridge_block_contract_request.json').read_text())['request']
if (hashlib.sha256(core_source.encode()).hexdigest() != prototypes['definition_sha256'] or
        hashlib.sha256(wrapper_source.encode()).hexdigest() != contract_request['wrapper_sha256']):
    raise RuntimeError('Bridge implementations changed.')
request = {
    'version': 'phase4-full-pipeline-insertion-smoke-v1', 'target': target, 'row': row,
    'layer': 5, 'prototype_request_key': prototype_request['request_key'],
    'checkpoint_sha256': checkpoint_hash, 'core_source': core_source, 'wrapper_source': wrapper_source,
    'local_contract_sha256': hashlib.sha256(contract_raw).hexdigest(),
    'identity_reference_sha256': reference_hash,
    'results_volume': target['runtime']['results_volume'],
    'scope': 'one block-5 initialization smoke; no training, no candidate selection, evaluation-only prompt',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
manifest = {'request': request, 'request_key': key}
manifest_path = run_dir / 'request.json'
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != manifest:
        raise RuntimeError('Insertion request differs; preserve artifacts.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
image_path = run_dir / 'student.png'
recipe_path = run_dir / 'student_recipe.json'
remote_path = f'/phase4/insertion/{key}/result.json'

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Insertion report belongs to other inputs.')
    if report['status'] != 'passed':
        raise RuntimeError('Saved insertion failed; no automatic retry: ' + report.get('error', 'unknown'))

report = None
if report_path.exists():
    report = json.loads(report_path.read_text())
    validate(report)
    if image_path.exists() and recipe_path.exists():
        for path, field in ((image_path, 'image_sha256'), (recipe_path, 'recipe_sha256')):
            if hashlib.sha256(path.read_bytes()).hexdigest() != report[field]:
                raise RuntimeError('Saved insertion artifact changed.')
        print('Reusing verified local insertion result; no Modal call or model forward.')
    else:
        report = None
if report is None:
    import modal
    runtime = target['runtime']
    volume = modal.Volume.from_name(request['results_volume'])
    try:
        packet = json.loads(b''.join(volume.read_file(remote_path)))
        print('Recovering durable remote insertion result; no GPU scheduled.')
    except FileNotFoundError:
        if report_path.exists() or image_path.exists() or recipe_path.exists():
            raise RuntimeError('Partial local result has no remote backup; do not regenerate.')
        app = modal.App('nihonga-phase4-insertion-smoke')
        weights = modal.Volume.from_name(runtime['weights_volume'])
        image = (modal.Image.debian_slim(python_version=runtime['python_version'])
                 .apt_install('git').pip_install(*runtime['packages'])
                 .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}))

        @app.function(image=image, gpu=runtime['gpu_requested'], serialized=True, timeout=1800,
                      max_containers=1, retries=0,
                      volumes={'/root/.cache/huggingface': weights.with_mount_options(read_only=True), '/phase3': volume})
        def run_insertion(expected, request_key, checkpoint_payload, reference_png):
            import base64
            import hashlib
            import importlib.metadata
            import inspect
            import io
            import json
            import time
            from datetime import datetime, timezone
            from pathlib import Path
            import numpy as np
            import torch
            from PIL import Image
            from diffusers import QwenImage21Pipeline
            from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache

            saved = Path('/phase3/phase4/insertion') / request_key / 'result.json'
            if saved.exists():
                return json.loads(saved.read_text())
            started_path = saved.with_name('started.json')
            saved.parent.mkdir(parents=True, exist_ok=True)
            if started_path.exists():
                raise RuntimeError('An earlier attempt started without a result; inspect instead of repeating.')
            started_path.write_text(json.dumps({'request_key': request_key, 'started_utc': datetime.now(timezone.utc).isoformat()}))
            volume.commit()
            measured = {'request_key': request_key, 'status': 'failed', 'layer': expected['layer'],
                        'recorded_utc': datetime.now(timezone.utc).isoformat()}
            packet = {'report': measured}
            try:
                if hashlib.sha256(checkpoint_payload).hexdigest() != expected['checkpoint_sha256']:
                    raise RuntimeError('Transferred checkpoint checksum differs.')
                if hashlib.sha256(reference_png).hexdigest() != expected['identity_reference_sha256']:
                    raise RuntimeError('Transferred reference checksum differs.')
                config, r = expected['target'], expected['row']
                measured['gpu'] = torch.cuda.get_device_name(0)
                if 'H100' not in measured['gpu']:
                    raise RuntimeError('Allocated GPU is not H100.')
                measured['packages'] = {p: importlib.metadata.version(p) for p in ('torch', 'diffusers', 'transformers', 'accelerate', 'pillow')}
                pipe = QwenImage21Pipeline.from_pretrained(config['model_repo'], revision=config['model_revision'],
                    dtype=getattr(torch, config['dtype']), local_files_only=True).to('cuda')
                pipe.set_progress_bar_config(disable=True)
                model = pipe.transformer
                model.eval()
                model.requires_grad_(False)
                if hashlib.sha256(Path(inspect.getfile(type(model))).read_bytes()).hexdigest() != config['transformer_source_sha256']:
                    raise RuntimeError('Remote transformer differs from Phase 3.')
                if (len(model.transformer_blocks) != config['num_layers'] or not model.config.causal_condition
                        or type(pipe.scheduler).__name__ != config['scheduler_class']
                        or json.loads(json.dumps(dict(pipe.scheduler.config), default=str)) != config['scheduler_config']):
                    raise RuntimeError('Teacher architecture or scheduler changed.')
                namespace = {}
                exec(compile(expected['core_source'], 'bridge_definition.py', 'exec'), namespace)
                exec(compile(expected['wrapper_source'], 'bridge_block_definition.py', 'exec'), namespace)
                checkpoint = torch.load(io.BytesIO(checkpoint_payload), weights_only=True, map_location='cpu')
                if checkpoint['request_key'] != expected['prototype_request_key'] or checkpoint['layer'] != expected['layer'] or checkpoint['trained']:
                    raise RuntimeError('Prototype checkpoint provenance differs.')
                bridge = namespace['ResidualBottleneckBridge'](4096, 256)
                bridge.load_state_dict(checkpoint['state_dict'], strict=True)
                if torch.count_nonzero(bridge.up.weight).item() != 0:
                    raise RuntimeError('Smoke check requires the original zero-up initialization.')
                total_before = sum(p.numel() for p in model.parameters())
                old, empty_cache = namespace['insert_bridge'](model, expected['layer'], bridge, QwenImage21KVCache)
                removed_parameters = sum(p.numel() for p in old.parameters())
                bridge_parameters = sum(p.numel() for p in bridge.parameters())
                total_after = sum(p.numel() for p in model.parameters())
                assert total_after == total_before - removed_parameters + bridge_parameters
                assert len(model.transformer_blocks) == config['num_layers']
                assert all(cache.k is None for cache in empty_cache.layer_caches)
                # No teacher cache has been constructed. Pipeline creates the student's cache.
                del old, empty_cache
                bridge.requires_grad_(True)
                calls = []
                def record_call(module, args, kwargs, output):
                    h = kwargs['hidden_states'] if 'hidden_states' in kwargs else args[0]
                    if not torch.equal(h, output):
                        raise RuntimeError('Initial bridge is not exactly identity in BF16.')
                    calls.append({'mode': kwargs.get('kv_cache_mode'), 'shape': list(h.shape),
                                  'dtype': str(h.dtype), 'exact_identity': True})
                hook = model.transformer_blocks[expected['layer']].register_forward_hook(record_call, with_kwargs=True)
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                started = time.perf_counter()
                with torch.inference_mode():
                    generated = pipe(prompt=r['prompt'], width=r['width'], height=r['height'],
                        num_inference_steps=r['num_inference_steps'], true_cfg_scale=r['true_cfg_scale'],
                        generator=torch.Generator(config['noise_device']).manual_seed(r['seed']), use_kv_cache=True).images[0]
                torch.cuda.synchronize()
                measured['generation_seconds'] = time.perf_counter() - started
                measured['peak_allocated_gib'] = torch.cuda.max_memory_allocated() / 2**30
                measured['peak_reserved_gib'] = torch.cuda.max_memory_reserved() / 2**30
                hook.remove()
                if (len(calls) != r['num_inference_steps'] or calls[0]['mode'] != 'extract'
                        or any(c['mode'] != 'cached' for c in calls[1:])):
                    raise RuntimeError('Unexpected bridge execution or cache modes.')
                if generated.size != (r['width'], r['height']):
                    raise RuntimeError('Unexpected output image dimensions.')
                buffer = io.BytesIO()
                generated.save(buffer, format='PNG')
                png = buffer.getvalue()
                reference = Image.open(io.BytesIO(reference_png)).convert('RGB')
                delta = np.asarray(generated.convert('RGB'), dtype=np.float64) - np.asarray(reference, dtype=np.float64)
                recipe = {'version': 'phase4-student-recipe-v1', 'request_key': request_key,
                          'teacher_model_repo': config['model_repo'], 'teacher_model_revision': config['model_revision'],
                          'teacher_config_sha256': config['config_sha256'], 'num_original_slots': config['num_layers'],
                          'surviving_transformer_blocks': config['num_layers'] - 1,
                          'replacement': {'slot': expected['layer'], 'checkpoint': 'data/phase4/bridge_block_05_initial.pt',
                                          'checkpoint_sha256': expected['checkpoint_sha256'], 'hidden_dim': 4096,
                                          'bottleneck_dim': 256, 'trained': False},
                          'core_sha256': hashlib.sha256(expected['core_source'].encode()).hexdigest(),
                          'wrapper_sha256': hashlib.sha256(expected['wrapper_source'].encode()).hexdigest(),
                          'transformer_source_sha256': config['transformer_source_sha256'],
                          'dtype': config['dtype'], 'cache_policy': 'fresh student cache; no teacher cache reuse',
                          'frozen': 'all inherited teacher parameters', 'trainable': 'bridge only',
                          'assembly': 'load pinned teacher, load checkpoint, insert_bridge at slot 5; keep original slot indexing'}
                recipe_text = json.dumps(recipe, indent=2) + '\n'
                measured.update(status='passed', calls=calls, image_sha256=hashlib.sha256(png).hexdigest(),
                                image_size=list(generated.size), recipe_sha256=hashlib.sha256(recipe_text.encode()).hexdigest(),
                                parameters_before=total_before, parameters_after=total_after,
                                removed_block_parameters=removed_parameters, bridge_parameters=bridge_parameters,
                                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                                identity_reference_sha256=expected['identity_reference_sha256'],
                                rgb8_mae=float(np.abs(delta).mean()), rgb8_rmse=float(np.sqrt((delta**2).mean())),
                                max_pixel_difference=int(np.abs(delta).max()), identical_reference_pixels=bool(np.all(delta == 0)),
                                teacher_regenerated=False, checkpoint_modified=False)
                packet.update(png_base64=base64.b64encode(png).decode(), recipe_text=recipe_text)
            except Exception as error:
                measured['error'] = f'{type(error).__name__}: {error}'
            temporary = saved.with_suffix('.tmp')
            temporary.write_text(json.dumps(packet, indent=2) + '\n')
            temporary.replace(saved)
            volume.commit()
            return packet

        with modal.enable_output(), app.run():
            packet = run_insertion.remote(request, key, checkpoint_bytes, reference_path.read_bytes())
    report = packet['report']
    if report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report['status'] == 'passed':
        for path, raw, field in ((image_path, base64.b64decode(packet['png_base64']), 'image_sha256'),
                                 (recipe_path, packet['recipe_text'].encode(), 'recipe_sha256')):
            if hashlib.sha256(raw).hexdigest() != report[field]:
                raise RuntimeError('Remote artifact checksum differs.')
            if path.exists():
                if path.read_bytes() != raw:
                    raise RuntimeError('Local artifact differs from backup.')
            else:
                temporary = path.with_suffix('.tmp')
                temporary.write_bytes(raw)
                temporary.replace(path)
    if report_path.exists():
        if json.loads(report_path.read_text()) != report:
            raise RuntimeError('Saved insertion report differs from backup.')
    else:
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(report, indent=2) + '\n')
    validate(report)
print('Full-pipeline insertion:', report['status'], '| block 5 |', len(report['calls']), 'BF16 forwards')
print('Parameters: removed block =', report['removed_block_parameters'], '| bridge =', report['bridge_parameters'])
print('Student transformer parameters:', report['parameters_after'], '| trainable:', report['trainable_parameters'])
print('Saved identity-bypass reference: MAE =', report['rgb8_mae'], '| RMSE =', report['rgb8_rmse'], '| identical pixels =', report['identical_reference_pixels'])
print('Smoke generation seconds:', report['generation_seconds'], '| peak allocated GiB:', report['peak_allocated_gib'])
print('Saved image, report and reproducible student recipe:', run_dir)
print('Untrained initialization smoke only; no quality recovery or speedup established.')


Reusing verified local insertion result; no Modal call or model forward.
Full-pipeline insertion: passed | block 5 | 40 BF16 forwards
Parameters: removed block = 218104064 | bridge = 2097152
Student transformer parameters: 6899117824 | trainable: 2097152
Saved identity-bypass reference: MAE = 0.0 | RMSE = 0.0 | identical pixels = True
Smoke generation seconds: 10.545286888999996 | peak allocated GiB: 36.492406368255615
Saved image, report and reproducible student recipe: data\phase4\insertion_block_05_smoke
Untrained initialization smoke only; no quality recovery or speedup established.


## What the full-model insertion results mean

**The block-5 wrapper works inside the pinned Qwen-Image 2.1 pipeline on an H100 in BF16.** One full 40-step generation completed at **1376 × 768 pixels**. The student's structure is **31 surviving transformer blocks plus one bridge**, still occupying the original 32 slots. We preserved slot numbering because the parent model's shared modulation and cache indexing use it. Block 5 is the integration example, not a selected winner; blocks 2, 3 and 4 remain candidates.

The bridge executed **40 times**: once in `extract` mode with **4,155 tokens**, then 39 times in `cached` mode with **4,128 target-image tokens**. The 27 extra tokens on the first call are the prompt prefix. Every call used BF16 and returned exactly its input, as expected from the zero up projection in the original untrained checkpoint. The full pipeline created its own student prefix cache; no teacher cache was reused.

| Quantity | Measured value | Meaning |
|---|---:|---|
| Removed transformer-block parameters | 218,104,064 | Original slot-5 block |
| Bridge parameters | 2,097,152 | Two bias-free projection matrices |
| Bridge / removed-block parameter ratio | 0.9615% | Bridge has about one hundredth as many parameters |
| Transformer parameters before insertion | 7,115,124,736 | Pinned teacher transformer, excluding text encoder and VAE |
| Transformer parameters after insertion | 6,899,117,824 | Student transformer retaining the shared parent projections |
| Transformer parameter reduction | 3.0359% | One block replacement changes the whole transformer by about 3% |
| Parameters marked trainable | 2,097,152 | Only the bridge; inherited transformer weights are frozen |

These are parameter counts, not operation counts or measured speedups. The wrapper contains no inherited attention or feed-forward parameters. Its residual operation starts as identity but still executes the two projection matrices.

**The generated image is pixel-identical to the saved Phase 3 block-5 identity-bypass image for the same meadow prompt, seed and sampling settings.** Mean absolute error and root mean squared error are both **0.0 on the 0–255 RGB scale**, and the maximum channel difference is **0**. The PNG checksum also matches. This confirms that the initial bridge reproduces identity bypass for this one trajectory. The comparison is against the bypassed model, not the intact teacher, and says nothing about recovered image quality. No teacher image was regenerated, no bridge was trained, and no initial checkpoint changed.

The smoke run took **10.55 seconds** and reached **36.49 GiB allocated / 38.42 GiB reserved** GPU memory. Timing includes first-run effects and per-call identity-verification hooks. This is a readiness measurement, not a matched latency benchmark; it does not support a speedup claim. Memory covers the loaded pipeline and is not training memory.

**Decision supported:** full-pipeline insertion is ready for teacher-supervised bridge work. **Still uncertain:** whether a learned bridge can reproduce the removed block, preserve prompt adherence and other capabilities, or improve measured latency enough to justify the architecture change. This run covers one text-only evaluation prompt with the standard attention processor; it does not establish full-pipeline condition-image behavior, all candidates, joint replacements, flex attention, compilation or training with checkpointing.

The saved `student_recipe.json` reconstructs this experimental student from the pinned teacher revision, unchanged initial bridge checkpoint and checksummed implementation. It records the original slot index, BF16 precision, frozen/trainable policy and fresh-cache requirement. This is a recipe referencing the cached teacher weights, not an independent full student weight export. The inherited shared modulation projection remains unchanged.

Results are in `data/phase4/insertion_block_05_smoke/`: `student.png`, `summary.json`, `student_recipe.json`, and `request.json`. The remote result packet backs up the image, recipe and measurements on the existing results volume. Initial checkpoints and teacher weights are preserved. A completed rerun of all three phase 4 code cells used only saved local artifacts, imported no Torch, Diffusers, Modal or network client, and changed no artifact.

![Block 5 initial bridge: saved meadow smoke image](data/phase4/insertion_block_05_smoke/student.png)
